# SynthWorlds: a tiny paired-world tutorial

This notebook accompanies [SynthWorlds (arXiv:2510.24427)](https://arxiv.org/abs/2510.24427). It loads a tiny sample from the [official public dataset](https://huggingface.co/datasets/kenqgu/SynthWorlds) and explains how parallel real-mapped (RM) and synthetic-mapped (SM) worlds separate factual familiarity from reasoning.

The key experimental quantity is **knowledge advantage**: `KA = performance(RM) - performance(SM)`. The paired tasks have the same graph structure, but only RM uses familiar real-world entities.

## 1. Load two official pairs

Hugging Face provides a small rows API for public datasets. Using it keeps this tutorial fast: we fetch two rows from each world instead of downloading the complete QA files. No package or account is required.

In [ ]:
import json
import re
import string
import urllib.parse
import urllib.request
from collections import Counter

DATASET = 'kenqgu/SynthWorlds'
ROWS_API = 'https://datasets-server.huggingface.co/rows'

def load_rows(config, offset=0, length=2):
    query = urllib.parse.urlencode({
        'dataset': DATASET,
        'config': config,
        'split': 'test',
        'offset': offset,
        'length': length,
    })
    request = urllib.request.Request(
        f'{ROWS_API}?{query}',
        headers={'User-Agent': 'SynthWorlds-Colab-Tutorial/1.0'},
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        payload = json.load(response)
    rows = [item['row'] for item in payload['rows']]
    if len(rows) != length:
        raise RuntimeError(f'Expected {length} rows from {config}, received {len(rows)}')
    return rows

qa_rm = load_rows('qa-rm')
qa_sm = load_rows('qa-sm')
print(f'Loaded {len(qa_rm)} RM rows and {len(qa_sm)} SM rows.')

## 2. Inspect the real released schema

Each row already contains its two gold documents, so printing the entire record would be noisy. This compact projection preserves the question, answer, graph type, document count, and decomposed questions. Notice that `mhop_qa` is stored as a JSON **string**, so we parse it once.

In [ ]:
def compact(row):
    metadata = json.loads(row['mhop_qa'])
    return {
        'instance_id': row['instance_id'],
        'query': row['query'],
        'gold_answers': row['gold_answers'],
        'question_graph_type': row['question_graph_type'],
        'expected_output_is_time': row['expected_output_is_time'],
        'gold_qids': row['gold_qids'],
        'number_of_gold_docs': len(row['gold_docs']),
        'decomposed_questions': metadata['decomposed_questions'],
    }

print('REAL-MAPPED EXAMPLE')
print(json.dumps(compact(qa_rm[0]), indent=2, ensure_ascii=False))
print('\nSYNTHETIC-MAPPED COUNTERPART')
print(json.dumps(compact(qa_sm[0]), indent=2, ensure_ascii=False))

## 3. Verify the controlled transformation

The pair ID is shared before the `-qa-rm` or `-qa-sm` suffix. The metadata also includes an `entity_rename_map`. Applying its old-to-new names to an RM query should produce the SM query while leaving the relation structure intact. Time answers are handled separately by the generator, so we do not expect an entity map to transform years.

In [ ]:
def pair_key(instance_id):
    return re.sub(r'-qa-(?:rm|sm)$', '', instance_id)

def rename_with_metadata(text, row):
    metadata = json.loads(row['mhop_qa'])
    replacements = [
        (record['old_name'], record['new_name'])
        for record in metadata['entity_rename_map'].values()
    ]
    # Longest first prevents a short name from modifying part of a longer one.
    for old_name, new_name in sorted(replacements, key=lambda pair: len(pair[0]), reverse=True):
        text = text.replace(old_name, new_name)
    return text

pair_checks = []
for rm, sm in zip(qa_rm, qa_sm):
    assert pair_key(rm['instance_id']) == pair_key(sm['instance_id'])
    mapped_query = rename_with_metadata(rm['query'], rm)
    mapped_answer = rename_with_metadata(rm['gold_answers'][0], rm)
    pair_checks.append({
        'pair': pair_key(rm['instance_id']),
        'rm_query': rm['query'],
        'sm_query': sm['query'],
        'entity_mapping_recreates_sm_query': mapped_query == sm['query'],
        'answer_check': (
            'time shifted separately' if rm['expected_output_is_time']
            else mapped_answer == sm['gold_answers'][0]
        ),
    })

print(json.dumps(pair_checks, indent=2, ensure_ascii=False))

## 4. See the two-hop evidence

A QA instance contains two supporting documents. The first establishes a bridge entity; the second follows another relation from that bridge to the answer. The decomposed questions printed above make this composition explicit.

In [ ]:
example = qa_rm[0]
print('QUESTION:', example['query'])
print('ANSWER:', example['gold_answers'][0])
for index, document in enumerate(example['gold_docs'], start=1):
    preview = document[:500].replace('\n', ' ')
    print(f'\nHOP {index} DOCUMENT PREVIEW:\n{preview}...')

## 5. Token F1 and the knowledge-advantage gap

The paper reports token-level answer F1 for multi-hop QA. We implement the familiar normalized overlap metric, then compute `KA = F1_RM - F1_SM`. The predictions below are deliberately constructed teaching examples—not outputs from a real model and not paper results.

In [ ]:
def normalize(text):
    text = text.casefold()
    text = ''.join(' ' if char in string.punctuation else char for char in text)
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    return ' '.join(text.split())

def token_f1(prediction, gold):
    predicted_tokens = normalize(prediction).split()
    gold_tokens = normalize(gold).split()
    if not predicted_tokens or not gold_tokens:
        return float(predicted_tokens == gold_tokens)
    overlap = sum((Counter(predicted_tokens) & Counter(gold_tokens)).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(predicted_tokens)
    recall = overlap / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

def row_f1(prediction, row):
    return max(token_f1(prediction, gold) for gold in row['gold_answers'])

def mean_f1(predictions, rows):
    return sum(row_f1(prediction, row) for prediction, row in zip(predictions, rows)) / len(rows)

# Toy closed-book behavior: knows the familiar RM answers, then repeats them in SM.
toy_closed_rm = [row['gold_answers'][0] for row in qa_rm]
toy_closed_sm = toy_closed_rm.copy()

# Toy oracle behavior: uses the correct evidence in each world.
toy_oracle_rm = [row['gold_answers'][0] for row in qa_rm]
toy_oracle_sm = [row['gold_answers'][0] for row in qa_sm]

def summarize_gap(label, rm_predictions, sm_predictions):
    rm_score = mean_f1(rm_predictions, qa_rm)
    sm_score = mean_f1(sm_predictions, qa_sm)
    return {'teaching_condition': label, 'F1_RM': rm_score, 'F1_SM': sm_score, 'KA': rm_score - sm_score}

results = [
    summarize_gap('toy closed-book memory', toy_closed_rm, toy_closed_sm),
    summarize_gap('toy oracle evidence use', toy_oracle_rm, toy_oracle_sm),
]
print(json.dumps(results, indent=2))

In the first toy condition, copying familiar-world answers fails after the entities and timestamps change, producing a positive knowledge advantage. In the oracle condition, correct world-specific evidence eliminates the gap. Real systems fall between these extremes; SynthWorlds measures how much of the gap remains under closed-book, RAG, iterative retrieval, or navigation conditions.

## 6. Loading the complete release

For full experiments, install `datasets` and load any official configuration:

```python
!pip -q install datasets
from datasets import load_dataset
qa_rm_full = load_dataset('kenqgu/SynthWorlds', 'qa-rm', split='test')
qa_sm_full = load_dataset('kenqgu/SynthWorlds', 'qa-sm', split='test')
```

The companion repository additionally provides the full evaluation stack. Reproducing its LLM and RAG baselines requires substantially more dependencies, model/API access, and compute than this tiny data tutorial.